# 3. Search check: the search links

The search check was done in a browser: automated Google searches are blocked with a CAPTCHA. This notebook builds the two links for each of the 31 stories in `data/search_check/search_stories.csv`:
- Google News search;
- Google Search's News tab.

Both are limited to the story's dates. Open each link, then record the top 10 results (rank, outlet, headline, time) in the format of `data/search_check/search_results.csv`. Mark results about a different story as `on_story = False`.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'collection' else Path.cwd()   # repository root
RAW, CACHE, OUT = ROOT / 'data' / 'raw', ROOT / 'cache', ROOT / 'outputs'
CACHE.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
from urllib.parse import quote_plus
import pandas as pd

stories = pd.read_csv(ROOT / 'data' / 'search_check' / 'search_stories.csv')
def gnews(r):
    lang = 'hl=fr-CA&gl=CA&ceid=CA:fr' if r.edition == 'fr-CA' else 'hl=en-CA&gl=CA&ceid=CA:en'
    return f'https://news.google.com/search?q={quote_plus(f"{r.query} after:{r.date_min} before:{r.date_max}")}&{lang}'
def mdy(d):
    t = pd.Timestamp(d)
    return f'{t.month}/{t.day}/{t.year}'
def gsearch(r):
    d0, d1 = mdy(r.date_min), mdy(r.date_max)
    lang = 'hl=fr&gl=ca' if r.edition == 'fr-CA' else 'hl=en&gl=ca'
    return f'https://www.google.com/search?q={quote_plus(r.query)}&tbm=nws&tbs=cdr:1,cd_min:{d0},cd_max:{d1}&{lang}'

links = stories.assign(google_news=[gnews(r) for r in stories.itertuples()],
                       google_search=[gsearch(r) for r in stories.itertuples()])
links[['story_id', 'edition', 'query', 'google_news', 'google_search']].to_csv(OUT / 'search_links.csv', index=False)
links[['story_id', 'query', 'google_news']].head()